In [105]:
"""
Complete Financial Analysis System
Combines ML predictions, fundamental data, and LLM analysis for comprehensive company evaluation
"""

import numpy as np
import pandas as pd
import joblib
import pickle
from tensorflow.keras.models import load_model
import rqdatac as rq
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# Initialize rqdatac
rq.init()

## MODEL LOADING AND PREDICTION FUNCTIONS

In [106]:

# ============================================
# 1. MODEL LOADING AND PREDICTION FUNCTIONS
# ============================================

class FinancialPredictor:
    """Handles all ML model predictions for cash flows and earnings"""
    
    def __init__(self, model_dir='./model'):
        self.model_dir = model_dir
        self.models = self._load_models()
        
    def _load_models(self):
        """Load all pre-trained models"""
        print("Loading ML models...")
        try:
            models = {
                # Cash Flow Models
                'ffcf_rf_model': joblib.load(f'{self.model_dir}/ffcf_random_forest_model.pkl'),
                'ffcf_rf_selector': joblib.load(f'{self.model_dir}/ffcf_rf_selector.pkl'),
                'ffcf_gbr_model': joblib.load(f'{self.model_dir}/ffcf_gradient_boosting_model.pkl'),
                'ffcf_lstm_model': load_model(f'{self.model_dir}/ffcf_lstm_model.h5'),
                'ffcf_x_scaler': joblib.load(f'{self.model_dir}/ffcf_x_scaler.pkl'),
                'ffcf_y_scaler': joblib.load(f'{self.model_dir}/ffcf_y_scaler.pkl'),
                
                # Earnings Models
                'earnings_rf_model': joblib.load(f'{self.model_dir}/earnings_random_forest_model.pkl'),
                'earnings_rf_selector': joblib.load(f'{self.model_dir}/earnings_rf_selector.pkl'),
                'earnings_gbr_model': joblib.load(f'{self.model_dir}/earnings_gradient_boosting_model.pkl'),
                'earnings_lstm_model': load_model(f'{self.model_dir}/earnings_lstm_model.h5'),
                'earnings_x_scaler': joblib.load(f'{self.model_dir}/earnings_x_scaler.pkl'),
                'earnings_y_scaler': joblib.load(f'{self.model_dir}/earnings_y_scaler.pkl'),
            }
            
            with open(f'{self.model_dir}/ffcf_model_config.pkl', 'rb') as f:
                models['ffcf_config'] = pickle.load(f)
                
            with open(f'{self.model_dir}/earnings_model_config.pkl', 'rb') as f:
                models['earnings_config'] = pickle.load(f)
                
            print("✓ All ML models loaded successfully")
            return models
            
        except Exception as e:
            print(f"Error loading models: {e}")
            raise
    
    def _prepare_features(self, financial_df: pd.DataFrame) -> np.ndarray:
        """Prepare features for model prediction"""
        # Group by year and take last value
        X = financial_df.groupby([
            pd.Grouper(level='order_book_id'),
            pd.Grouper(level='date', freq='Y')
        ]).last().fillna(0)
        
        X = X.reorder_levels(['order_book_id', 'date'])
        X = X.tail(5)
        
        # Flatten for ML models
        prediction_input = X.values.flatten().reshape(1, -1)
        return prediction_input
    
    def predict_cash_flows(self, financial_df: pd.DataFrame) -> Dict[str, np.ndarray]:
        """Predict 5-year cash flows using all models"""
        features = self._prepare_features(financial_df)
        
        predictions = {}
        
        # Random Forest
        features_selected = self.models['ffcf_rf_selector'].transform(features)
        predictions['random_forest'] = self.models['ffcf_rf_model'].predict(features_selected)[0]
        
        # Gradient Boosting
        predictions['gradient_boosting'] = self.models['ffcf_gbr_model'].predict(features)[0]
        
        # LSTM
        features_scaled = self.models['ffcf_x_scaler'].transform(features)
        n_features = features_scaled.shape[1] // 5
        features_lstm = features_scaled.reshape(1, 5, n_features)
        lstm_pred_scaled = self.models['ffcf_lstm_model'].predict(features_lstm, verbose=0)
        predictions['lstm'] = self.models['ffcf_y_scaler'].inverse_transform(lstm_pred_scaled)[0]
        
        # Ensemble (average of all models)
        ensemble = np.mean([predictions['random_forest'], 
                           predictions['gradient_boosting'], 
                           predictions['lstm']], axis=0)
        predictions['ensemble'] = ensemble
        
        return predictions
    
    def predict_earnings(self, financial_df: pd.DataFrame) -> Dict[str, np.ndarray]:
        """Predict 5-year earnings per share using all models"""
        features = self._prepare_features(financial_df)
        
        predictions = {}
        
        # Random Forest
        features_selected = self.models['earnings_rf_selector'].transform(features)
        predictions['random_forest'] = self.models['earnings_rf_model'].predict(features_selected)[0]
        
        # Gradient Boosting
        predictions['gradient_boosting'] = self.models['earnings_gbr_model'].predict(features)[0]
        
        # LSTM
        features_scaled = self.models['earnings_x_scaler'].transform(features)
        n_features = features_scaled.shape[1] // 5
        features_lstm = features_scaled.reshape(1, 5, n_features)
        lstm_pred_scaled = self.models['earnings_lstm_model'].predict(features_lstm, verbose=0)
        predictions['lstm'] = self.models['earnings_y_scaler'].inverse_transform(lstm_pred_scaled)[0]
        
        # Ensemble
        ensemble = np.mean([predictions['random_forest'], 
                           predictions['gradient_boosting'], 
                           predictions['lstm']], axis=0)
        predictions['ensemble'] = ensemble
        
        return predictions

## Fundamental Data Collection

In [107]:

# ============================================
# 2. FUNDAMENTAL DATA COLLECTION
# ============================================

class FundamentalDataCollector:
    """Collects and processes fundamental financial data"""
    
    def __init__(self):
        self.factors = [
            # Revenue & Profitability
            'revenue', 'operating_revenue', 'net_profit', 'net_profit_parent_company',
            'ebit', 'ebitda', 'profit_before_tax',
            
            # Ratios
            'return_on_equity', 'return_on_asset', 'debt_to_equity', 'current_ratio',
            'revenue_growth_rate', 'net_profit_growth_rate',
            
            # Per share metrics
            'basic_earnings_per_share', 'book_value_per_share', 'free_cash_flow_company_per_share',
            
            # Balance sheet
            'total_assets', 'total_equity', 'current_assets', 'current_liabilities',
            'long_term_loans', 'short_term_loans', 'cash_equivalent',
            'net_accts_receivable', 'inventory', 'net_fixed_assets',
            'intangible_assets', 'total_liabilities',
        ]
        
        # Valuation factors for DCF
        self.valuation_factors = [
            'pe_ratio', 'pb_ratio', 'ps_ratio', 'pcf_ratio'
        ]

        # Earnings prediction factors        
        self.earnings_factors = [
            'basic_earnings_per_share',
            'net_profit',
            'net_profitTTM',
            'net_profit_parent_company',
            'ebit',
            'ebitda',
            'return_on_equity',
            'return_on_asset'

            # revenue
            'revenue',
            'revenue_growth_rate',
            'net_profit_growth_rate',
            
            # valuation
            'book_value_per_share',
            'book_value_per_share_ttm',

            # working capital
            'working_capital',                     # Net liquidity
            'working_capital_ttm',                 # TTM working capital
            'inventory',                           # Inventory levels
            'net_accts_receivable',                # Receivables collection
            'current_assets',                      # Total current assets
            'current_liabilities',                 # Total current liabilities
            'cash_equivalent',                     # Cash position

            # investment and fixed assets
            'net_fixed_assets',                    # Asset base for earnings
            'depreciation_and_amortization',       # Non-cash expense affecting net income

            # balance sheet core metrics
            'total_assets',                        # Size basis
            'total_equity',                        # Book value base
            'equity_parent_company',               # Parent equity
            'intangible_assets',                   # Goodwill/intangibles (earnings quality)

            # liabilites structure
            'short_term_loans',                    # Short-term debt
            'long_term_loans',                     # Long-term debt
            'bond_payable',                        # Bond obligations
            'long_term_liabilities_due_one_year',  # Current portion of long-term debt

            # leverage/liquidity
            'debt_to_equity',
            'current_ratio',
        ]

        # Cash flow prediction factors
        self.cash_flow_factors = [
            # revenue
            'revenue',
            'operating_revenue',
            'net_operating_revenue',
            
            # valuation
            'book_value_per_share',
            'book_value_per_share_lf',
            'book_value_per_share_ttm',

            # profitability
            'return_on_equity',
            'return_on_asset',
            "profit_before_tax",
            "net_profit",
            "net_profit_parent_company",
            "ebit",

            # leverage/liquidity
            'debt_to_equity',
            'current_ratio',
            
            # growth
            'revenue_growth_rate',
            'net_profit_growth_rate',

            # free cash flow
            'free_cash_flow_company_per_share',
            'free_cash_flow_company_per_share_ttm',

            # working capital
            "working_capital",
            "working_capital_lf",
            "working_capital_ttm",

            # investment
            "net_current_investment",

            # assets and liabilities
            "current_assets",
            "current_liabilities",

            "depreciation_and_amortization",

            "ebitda",

            # Balance
            "cash_equivalent", 
            "bill_receivable", 
            "net_accts_receivable", 
            "inventory", 
            "long_term_equity_investment", 
            "net_long_term_equity_investment", 
            "net_fixed_assets", 
            "intangible_assets", 
            "short_term_loans", 
            "long_term_liabilities_due_one_year", 
            "long_term_loans", 
            "bond_payable", 
            "long_term_payable", 
            "total_assets", 
            "equity_parent_company", 
            "total_equity",
        ]
    
    def get_order_book_id(self, company_name: str) -> Optional[str]:
        """Get order_book_id from company name"""
        all_stocks = rq.all_instruments(type='CS')
        
        # Try exact match first
        stock = all_stocks[all_stocks['symbol'] == company_name]
        if not stock.empty:
            return stock['order_book_id'].values[0]
        
        # Try partial match
        stock = all_stocks[all_stocks['abbrev_symbol'].str.contains(company_name, na=False)]
        if not stock.empty:
            return stock['order_book_id'].values[0]
        
        return None
    
    def collect_fundamental_data(self, order_book_id: str) -> pd.DataFrame:
        """Collect latest 5 years of fundamental data"""
        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = '2022-01-01'
        
        # Get fundamental factors
        data = rq.get_factor(order_book_id, self.factors, 
                            start_date=start_date, end_date=end_date)
        data = data.fillna(0)
        
        # Resample to yearly
        yearly_data = data.groupby([
            pd.Grouper(level='order_book_id'),
            pd.Grouper(level='date', freq='Y')
        ]).last()
        
        return yearly_data
    
    def collect_earnings_prediction_data(self, order_book_id: str) -> pd.DataFrame:
        """Collect latest 5 years of fundamental data"""
        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = '2022-01-01'
        
        # Get fundamental factors
        data = rq.get_factor(order_book_id, self.earnings_factors, 
                            start_date=start_date, end_date=end_date)
        data = data.fillna(0)
        
        # Resample to yearly
        yearly_data = data.groupby([
            pd.Grouper(level='order_book_id'),
            pd.Grouper(level='date', freq='Y')
        ]).last()
        
        return yearly_data
    
    def collect_cash_flow_prediction_data(self, order_book_id: str) -> pd.DataFrame:
        """Collect latest 5 years of fundamental data"""
        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = '2022-01-01'
        
        # Get fundamental factors
        data = rq.get_factor(order_book_id, self.cash_flow_factors, 
                            start_date=start_date, end_date=end_date)
        data = data.fillna(0)
        
        # Resample to yearly
        yearly_data = data.groupby([
            pd.Grouper(level='order_book_id'),
            pd.Grouper(level='date', freq='Y')
        ]).last()
        
        return yearly_data
    
    def collect_valuation_data(self, order_book_id: str) -> pd.DataFrame:
        """Collect latest valuation ratios"""
        end_date = datetime.now().strftime('%Y-%m-%d')
        start_date = '2022-01-01'
        
        data = rq.get_factor(order_book_id, self.valuation_factors,
                            start_date=start_date, end_date=end_date)
        data = data.fillna(0)
        
        # Get latest values
        latest = data.groupby([
            pd.Grouper(level='order_book_id'),
            pd.Grouper(level='date', freq='M')
        ]).last().tail(1)
        
        return latest

## DCF Valuation

In [108]:

# ============================================
# 3. DCF VALUATION
# ============================================

class DCFValuator:
    """Performs Discounted Cash Flow valuation"""
    
    def __init__(self, discount_rate: float = 0.12, terminal_growth: float = 0.03):
        self.discount_rate = discount_rate
        self.terminal_growth = terminal_growth
    
    def calculate_terminal_value(self, final_cf: float) -> float:
        """Calculate terminal value using Gordon Growth Model"""
        return final_cf * (1 + self.terminal_growth) / (self.discount_rate - self.terminal_growth)
    
    def calculate_dcf(self, cash_flows: np.ndarray) -> float:
        """Calculate DCF value from 5-year cash flow projections"""
        pv = 0
        
        # Discount each year's cash flow
        for i, cf in enumerate(cash_flows[:5], start=1):
            pv += cf / (1 + self.discount_rate) ** i
        
        # Add terminal value
        terminal_value = self.calculate_terminal_value(cash_flows[4])
        pv += terminal_value / (1 + self.discount_rate) ** 5
        
        return pv

## Moat Analysis

In [109]:

# ============================================
# 4. MOAT ANALYSIS
# ============================================

class MoatAnalyzer:
    """Analyzes company's competitive moat based on fundamentals"""
    
    def analyze_moat(self, fundamental_data: pd.DataFrame) -> Dict:
        """Analyze the durability and size of the company's moat"""
        latest = fundamental_data.tail(1).iloc[0]
        
        moat_indicators = {
            'profitability': {
                'roe': latest.get('return_on_equity', 0),
                'roa': latest.get('return_on_asset', 0),
                'net_margin': latest.get('net_profit', 0) / latest.get('revenue', 1) if latest.get('revenue', 0) > 0 else 0,
            },
            'efficiency': {
                'receivables_turnover': latest.get('revenue', 0) / latest.get('net_accts_receivable', 1) if latest.get('net_accts_receivable', 0) > 0 else 0,
                'inventory_turnover': latest.get('revenue', 0) / latest.get('inventory', 1) if latest.get('inventory', 0) > 0 else 0,
                'asset_turnover': latest.get('revenue', 0) / latest.get('total_assets', 1) if latest.get('total_assets', 0) > 0 else 0,
            },
            'financial_health': {
                'debt_to_equity': latest.get('debt_to_equity', 0),
                'current_ratio': latest.get('current_ratio', 0),
                'revenue_growth': latest.get('revenue_growth_rate', 0),
                'profit_growth': latest.get('net_profit_growth_rate', 0),
            }
        }
        
        # Calculate moat score (0-10)
        score = 0
        
        # Profitability (max 4 points)
        if moat_indicators['profitability']['roe'] > 15:
            score += 2
        elif moat_indicators['profitability']['roe'] > 10:
            score += 1
            
        if moat_indicators['profitability']['net_margin'] > 0.15:
            score += 2
        elif moat_indicators['profitability']['net_margin'] > 0.10:
            score += 1
        
        # Growth (max 3 points)
        if moat_indicators['financial_health']['revenue_growth'] > 0.15:
            score += 1.5
        elif moat_indicators['financial_health']['revenue_growth'] > 0.05:
            score += 0.75
            
        if moat_indicators['financial_health']['profit_growth'] > 0.15:
            score += 1.5
        elif moat_indicators['financial_health']['profit_growth'] > 0.05:
            score += 0.75
        
        # Financial health (max 3 points)
        if moat_indicators['financial_health']['debt_to_equity'] < 0.5:
            score += 1.5
        elif moat_indicators['financial_health']['debt_to_equity'] < 1.0:
            score += 0.75
            
        if moat_indicators['financial_health']['current_ratio'] > 2:
            score += 1.5
        elif moat_indicators['financial_health']['current_ratio'] > 1:
            score += 0.75
        
        # Determine moat size and durability
        if score >= 7:
            moat_size = "Wide Moat"
            durability = "Very Durable"
            description = "Exceptional competitive advantages with high barriers to entry"
        elif score >= 5:
            moat_size = "Narrow Moat"
            durability = "Moderately Durable"
            description = "Good competitive position but with some vulnerabilities"
        elif score >= 3:
            moat_size = "No Moat"
            durability = "Limited Durability"
            description = "Weak competitive advantages, facing significant competition"
        else:
            moat_size = "Negative Moat"
            durability = "Unstable"
            description = "Disadvantages relative to competitors"
        
        return {
            'score': score,
            'moat_size': moat_size,
            'durability': durability,
            'description': description,
            'indicators': moat_indicators
        }

## Health Assessment

In [110]:

# ============================================
# 5. HEALTH ASSESSMENT
# ============================================

class HealthAssessor:
    """Assesses overall financial health of the company"""
    
    def assess_health(self, fundamental_data: pd.DataFrame, 
                      predictions: Dict) -> Dict:
        """Comprehensive health assessment"""
        latest = fundamental_data.tail(1).iloc[0]
        
        health_metrics = {
            'profitability_health': self._assess_profitability(latest),
            'liquidity_health': self._assess_liquidity(latest),
            'leverage_health': self._assess_leverage(latest),
            'growth_health': self._assess_growth(latest, predictions),
            'valuation_health': self._assess_valuation(latest),
        }
        
        # Calculate overall health score (0-100)
        scores = [v['score'] for v in health_metrics.values()]
        overall_score = np.mean(scores)
        
        if overall_score >= 80:
            overall_status = "Excellent"
            recommendation = "Strong Buy"
        elif overall_score >= 65:
            overall_status = "Good"
            recommendation = "Buy"
        elif overall_score >= 50:
            overall_status = "Fair"
            recommendation = "Hold"
        elif overall_score >= 35:
            overall_status = "Poor"
            recommendation = "Sell"
        else:
            overall_status = "Critical"
            recommendation = "Strong Sell"
        
        return {
            'overall_score': overall_score,
            'overall_status': overall_status,
            'recommendation': recommendation,
            'metrics': health_metrics,
            'warning_signs': self._identify_warning_signs(latest, health_metrics)
        }
    
    def _assess_profitability(self, data: pd.Series) -> Dict:
        score = 0
        roe = data.get('return_on_equity', 0)
        roa = data.get('return_on_asset', 0)
        
        if roe > 15:
            score += 40
        elif roe > 10:
            score += 30
        elif roe > 5:
            score += 20
        elif roe > 0:
            score += 10
        
        if roa > 5:
            score += 30
        elif roa > 3:
            score += 20
        elif roa > 0:
            score += 10
        
        return {'score': score, 'roe': roe, 'roa': roa}
    
    def _assess_liquidity(self, data: pd.Series) -> Dict:
        score = 0
        current_ratio = data.get('current_ratio', 0)
        cash = data.get('cash_equivalent', 0)
        
        if current_ratio > 2:
            score += 50
        elif current_ratio > 1.5:
            score += 35
        elif current_ratio > 1:
            score += 20
        elif current_ratio > 0.5:
            score += 10
        
        if cash > 0:
            score += 20
        
        return {'score': score, 'current_ratio': current_ratio, 'cash': cash}
    
    def _assess_leverage(self, data: pd.Series) -> Dict:
        score = 0
        debt_to_equity = data.get('debt_to_equity', 0)
        
        if debt_to_equity < 0.3:
            score += 50
        elif debt_to_equity < 0.5:
            score += 40
        elif debt_to_equity < 1:
            score += 25
        elif debt_to_equity < 2:
            score += 10
        
        return {'score': score, 'debt_to_equity': debt_to_equity}
    
    def _assess_growth(self, data: pd.Series, predictions: Dict) -> Dict:
        score = 0
        revenue_growth = data.get('revenue_growth_rate', 0)
        profit_growth = data.get('net_profit_growth_rate', 0)
        
        # Historical growth
        if revenue_growth > 0.15:
            score += 20
        elif revenue_growth > 0.05:
            score += 15
        elif revenue_growth > 0:
            score += 10
        
        if profit_growth > 0.15:
            score += 20
        elif profit_growth > 0.05:
            score += 15
        elif profit_growth > 0:
            score += 10
        
        # Future growth from ML predictions
        if 'ensemble' in predictions:
            avg_future_earnings = np.mean(predictions['ensemble'][:3])
            current_eps = data.get('basic_earnings_per_share', 1)
            if current_eps > 0:
                future_growth_rate = (avg_future_earnings / current_eps - 1)
                if future_growth_rate > 0.1:
                    score += 20
                elif future_growth_rate > 0.05:
                    score += 15
                elif future_growth_rate > 0:
                    score += 10
        
        return {'score': score, 'revenue_growth': revenue_growth, 'profit_growth': profit_growth}
    
    def _assess_valuation(self, data: pd.Series) -> Dict:
        score = 70  # Default neutral score
        pe = data.get('pe_ratio', 0)
        
        if 10 < pe < 20:
            score = 80
        elif pe <= 10:
            score = 90
        elif pe >= 30:
            score = 50
        elif pe >= 25:
            score = 60
        
        return {'score': score, 'pe_ratio': pe}
    
    def _identify_warning_signs(self, data: pd.Series, metrics: Dict) -> List[str]:
        warnings = []
        
        if data.get('debt_to_equity', 0) > 1.5:
            warnings.append("High leverage: Debt-to-equity ratio exceeds 1.5x")
        
        if data.get('current_ratio', 0) < 1:
            warnings.append("Liquidity concern: Current ratio below 1.0")
        
        if data.get('return_on_equity', 0) < 5:
            warnings.append("Weak profitability: ROE below 5%")
        
        if data.get('revenue_growth_rate', 0) < 0:
            warnings.append("Declining revenue: Negative growth rate")
        
        if data.get('net_profit_growth_rate', 0) < 0:
            warnings.append("Declining profits: Negative earnings growth")
        
        return warnings

## LLM Integration

In [ ]:

# ============================================
# 6. LLM INTEGRATION (DeepSeek/OpenAI)
# ============================================

from openai import OpenAI
import os

DEFAULT_API_KEY = ""
DEFAULT_BASE_URL = "https://api.deepseek.com"
DEFAULT_MODEL = "gpt-5-nano"

class LLMAnalyzer:
    """Handles LLM-powered analysis and report generation"""
    
    def __init__(
            self,
            api_key: Optional[str] = None,
            base_url: Optional[str] = None,
            model: Optional[str] = None
    ):
        self.api_key = api_key or os.environ.get("DEEPSEEK_API_KEY", DEFAULT_API_KEY)
        self.base_url = base_url or os.environ.get("DEEPSEEK_BASE_URL", DEFAULT_BASE_URL)
        self.model = model or DEFAULT_MODEL
        
        if not self.api_key:
            print("Warning: No API key found. LLM features will not work.")
            self.client = None
        else:
            # self.client = OpenAI(api_key=self.api_key, base_url=self.base_url)
            self.client = OpenAI(api_key=self.api_key)
    
    def generate_report(self, company_name: str, order_book_id: str,
                       fundamental_data: pd.DataFrame,
                       cash_flow_predictions: Dict,
                       earnings_predictions: Dict,
                       dcf_results: Dict,
                       moat_analysis: Dict,
                       health_assessment: Dict) -> str:
        """Generate comprehensive financial analysis report"""
        
        if not self.client:
            return "LLM not configured. Please set DEEPSEEK_API_KEY environment variable."
        
        # Prepare data for LLM
        latest_data = fundamental_data.tail(1).iloc[0]
        
        prompt = f"""You are a professional financial analyst. Generate a comprehensive investment analysis report for {company_name} ({order_book_id}).

## CURRENT FINANCIAL POSITION (Latest Year):
- Revenue: {latest_data.get('revenue', 0):,.0f}
- Net Profit: {latest_data.get('net_profit', 0):,.0f}
- EPS: {latest_data.get('basic_earnings_per_share', 0):.2f}
- ROE: {latest_data.get('return_on_equity', 0):.1f}%
- ROA: {latest_data.get('return_on_asset', 0):.1f}%
- Debt/Equity: {latest_data.get('debt_to_equity', 0):.2f}
- Current Ratio: {latest_data.get('current_ratio', 0):.2f}
- Revenue Growth: {latest_data.get('revenue_growth_rate', 0):.1f}%
- Profit Growth: {latest_data.get('net_profit_growth_rate', 0):.1f}%
- P/E Ratio: {latest_data.get('pe_ratio', 0):.1f}

## ML MODEL PREDICTIONS (Next 5 Years):

### Cash Flow per Share Predictions (Ensemble):
{', '.join([f'Year {i+1}: ${cf:.2f}' for i, cf in enumerate(cash_flow_predictions['ensemble'])])}

### Earnings per Share Predictions (Ensemble):
{', '.join([f'Year {i+1}: ${eps:.2f}' for i, eps in enumerate(earnings_predictions['ensemble'])])}

## DCF VALUATION:
- DCF Value per Share: ${dcf_results:.2f}

## MOAT ANALYSIS:
- Moat Score: {moat_analysis['score']:.1f}/10
- Moat Size: {moat_analysis['moat_size']}
- Durability: {moat_analysis['durability']}
- Description: {moat_analysis['description']}

## HEALTH ASSESSMENT:
- Overall Health Score: {health_assessment['overall_score']:.1f}/100
- Health Status: {health_assessment['overall_status']}
- Investment Recommendation: {health_assessment['recommendation']}

### Warning Signs:
{chr(10).join(['- ' + w for w in health_assessment['warning_signs']]) if health_assessment['warning_signs'] else '- No major warning signs identified'}

## YOUR TASK:
Write a detailed investment analysis report that covers:

1. **Executive Summary** - Brief overview of company's position and outlook
2. **Financial Health Analysis** - Deep dive into profitability, liquidity, leverage, and growth
3. **Competitive Moat Assessment** - Analysis of the company's competitive advantages and durability
4. **Valuation Analysis** - Whether the stock is undervalued, fair, or overvalued based on DCF
5. **Future Outlook** - Based on ML model predictions and industry context
6. **Risk Factors** - Key risks to consider
7. **Investment Recommendation** - Clear buy/sell/hold recommendation with price target
8. **Key Metrics to Monitor** - What to watch in future quarters

Be specific, data-driven, and provide clear reasoning for your conclusions. Use bullet points and sections for readability. Be honest about limitations and uncertainties. Make sure to reference the actual numbers provided above."""

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are an expert financial analyst with deep experience in valuation, competitive analysis, and investment research."},
                    {"role": "user", "content": prompt}
                ],
                temperature=1.0,
                # max_tokens=4000
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error generating LLM report: {str(e)}"
    
    def chat(self, messages: List[Dict], user_message: str) -> str:
        """Interactive chat with LLM about the analysis"""
        if not self.client:
            return "LLM not configured. Please set DEEPSEEK_API_KEY environment variable."
        
        messages.append({"role": "user", "content": user_message})
        
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=1.0,
                # max_tokens=2000
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error: {str(e)}"

## Main Integration System

In [112]:

# ============================================
# 7. MAIN INTEGRATION SYSTEM
# ============================================

DISCOUNT_RATE=0.12
TERMINAL_GROWTH=0.03

class FinancialAnalysisSystem:
    """Main system orchestrating all components"""
    
    def __init__(self, model_dir='./model'):
        print("Initializing Financial Analysis System...")
        self.predictor = FinancialPredictor(model_dir)
        self.data_collector = FundamentalDataCollector()
        self.dcf_valuator = DCFValuator(DISCOUNT_RATE, TERMINAL_GROWTH)
        self.moat_analyzer = MoatAnalyzer()
        self.health_assessor = HealthAssessor()
        self.llm_analyzer = LLMAnalyzer()
        print("System ready!\n")
    
    def analyze_company(self, company_name: str) -> Tuple[Dict, str, List]:
        """Complete analysis pipeline for a company"""
        
        print(f"\n{'='*60}")
        print(f"Analyzing {company_name}...")
        print(f"{'='*60}\n")
        
        # Step 1: Get order_book_id
        print("1. Looking up company...")
        order_book_id = self.data_collector.get_order_book_id(company_name)
        if not order_book_id:
            raise ValueError(f"Company '{company_name}' not found")
        print(f"   Found: {order_book_id}\n")
        
        # Step 2: Collect fundamental data
        print("2. Collecting fundamental data...")
        fundamental_data = self.data_collector.collect_fundamental_data(order_book_id)
        if fundamental_data.empty:
            raise ValueError("No fundamental data available")
        print(f"   Collected {len(fundamental_data)} years of data\n")
        print(f"   Fundamental Data: {fundamental_data}")
        
        # Step 3: Get latest valuation and prediction data
        print("3. Collecting valuation and prediction data...")
        valuation_data = self.data_collector.collect_valuation_data(order_book_id)
        if not valuation_data.empty:
            # Merge valuation data into fundamental data
            for col in valuation_data.columns:
                if col in fundamental_data.columns:
                    fundamental_data[col] = valuation_data[col].iloc[0] if not valuation_data.empty else 0
        print("   Valuation data collected\n")
        print(f"   Valuation Data: {valuation_data}")

        earnings_prediction_data = self.data_collector.collect_earnings_prediction_data(order_book_id)
        print("   Earnings prediction data collected\n")

        cash_flow_prediction_data = self.data_collector.collect_cash_flow_prediction_data(order_book_id)
        print("   Cash flow prediction data collected\n")
        
        # Step 4: Make ML predictions
        print("4. Generating ML predictions...")
        cash_flow_preds = self.predictor.predict_cash_flows(cash_flow_prediction_data)
        earnings_preds = self.predictor.predict_earnings(earnings_prediction_data)
        print("   Predictions complete\n")
        
        # Step 5: Calculate DCF
        print("5. Performing DCF valuation...")
        dcf_value = self.dcf_valuator.calculate_dcf(cash_flow_preds['ensemble'])
        print(f"   DCF Value: ${dcf_value:.2f}")
        
        # Step 6: Moat analysis
        print("6. Analyzing competitive moat...")
        moat_analysis = self.moat_analyzer.analyze_moat(fundamental_data)
        print(f"   Moat Score: {moat_analysis['score']:.1f}/10")
        print(f"   Moat Size: {moat_analysis['moat_size']}\n")
        
        # Step 7: Health assessment
        print("7. Assessing financial health...")
        health_assessment = self.health_assessor.assess_health(fundamental_data, earnings_preds)
        print(f"   Health Score: {health_assessment['overall_score']:.1f}/100")
        print(f"   Recommendation: {health_assessment['recommendation']}\n")
        
        # Step 8: Generate LLM report
        print("8. Generating AI-powered report...")
        report = self.llm_analyzer.generate_report(
            company_name, order_book_id, fundamental_data,
            cash_flow_preds, earnings_preds, dcf_value,
            moat_analysis, health_assessment
        )
        print("   Report generated!\n")
        
        # Package results for chat
        chat_context = [
            {"role": "system", "content": f"You are analyzing {company_name} ({order_book_id}). You have access to the following analysis results."},
            {"role": "system", "content": f"Health Assessment: {health_assessment}"},
            {"role": "system", "content": f"Moat Analysis: {moat_analysis}"},
            {"role": "system", "content": f"DCF Value: {dcf_value}"},
            {"role": "system", "content": f"Cash Flow Predictions: {cash_flow_preds['ensemble']}"},
            {"role": "system", "content": f"Earnings Predictions: {earnings_preds['ensemble']}"},
        ]
        
        results = {
            'company_name': company_name,
            'order_book_id': order_book_id,
            'fundamental_data': fundamental_data,
            'cash_flow_predictions': cash_flow_preds,
            'earnings_predictions': earnings_preds,
            'dcf_value': dcf_value,
            'moat_analysis': moat_analysis,
            'health_assessment': health_assessment
        }
        
        return results, report, chat_context
    
    def interactive_chat(self, chat_context: List[Dict]):
        """Interactive Q&A session with the LLM"""
        print("\n" + "="*60)
        print("INTERACTIVE Q&A SESSION")
        print("="*60)
        print("Ask questions about the company's financial health, valuation, or investment prospects.")
        print("Type 'quit' to exit.\n")
        
        messages = chat_context.copy()
        
        while True:
            question = input("\n❓ Your question: ").strip()
            if question.lower() in ['quit', 'exit', 'q']:
                print("\nThank you for using the Financial Analysis System!")
                break
            
            if not question:
                continue
            
            print("\n🤔 Analyzing...\n")
            response = self.llm_analyzer.chat(messages, question)
            print(f"💡 Answer:\n{response}\n")
            
            # Update context with the Q&A
            messages.append({"role": "user", "content": question})
            messages.append({"role": "assistant", "content": response})

## Main Execution

In [113]:

# ============================================
# 8. MAIN EXECUTION
# ============================================

def main():
    """
    Run the Financial Analysis System
    """
    # Initialize the system
    system = FinancialAnalysisSystem(model_dir='./model')
    
    # Get company name from user
    print("\n" + "="*60)
    company_name = input("Enter company name (e.g., '中国平安' or 'Ping An'): ").strip()
    
    # try:
    # Run complete analysis
    results, report, chat_context = system.analyze_company(company_name)
    
    # Display the report
    print("\n" + "="*60)
    print("INVESTMENT ANALYSIS REPORT")
    print("="*60)
    print(report)
    
    # Ask if user wants interactive Q&A
    print("\n" + "="*60)
    interactive = input("\nDo you want to ask follow-up questions? (yes/no): ").strip().lower()
    
    if interactive in ['yes', 'y']:
        system.interactive_chat(chat_context)
    else:
        print("\nThank you for using the Financial Analysis System!")
            
    # except Exception as e:
    #     print(f"\n❌ Error: {e}")
    #     print("\nPossible issues:")
    #     print("1. Check if company name is correct")
    #     print("2. Verify rqdatac is properly initialized")
    #     print("3. Ensure all model files exist in the directory")
    #     print("4. Check your internet connection")

In [114]:
main()

Initializing Financial Analysis System...
Loading ML models...
✓ All ML models loaded successfully
System ready!



Analyzing 中芯国际...

1. Looking up company...
   Found: 688981.XSHG

2. Collecting fundamental data...
   Collected 5 years of data

   Fundamental Data:                                revenue  operating_revenue    net_profit  \
order_book_id date                                                        
688981.XSHG   2022-12-31  3.776356e+10       3.776356e+10  1.162450e+10   
              2023-12-31  3.309824e+10       3.309824e+10  4.801253e+09   
              2024-12-31  4.187872e+10       4.187872e+10  3.232660e+09   
              2025-12-31  4.951042e+10       4.951042e+10  5.770359e+09   
              2026-12-31  1.761722e+10       1.761722e+10  1.593998e+09   

                          net_profit_parent_company  ebit  ebitda  \
order_book_id date                                                  
688981.XSHG   2022-12-31               9.389507e+09   0.0     0.0   